# Paired Statistical Comparisons and Bootstrap Intervals

This notebook documents paired final-model comparison across completed seeds. It recomputes evaluation metrics, applies McNemar tests with Holm adjustment to pre-specified contrasts, and derives bootstrap confidence intervals without fitting or selecting a new model.


## Release Boundary

This restricted code-inspection copy retains the analytical logic but does not include Banjarbaru review data, completed manual-audit records, row-level predictions, bootstrap replicates, or private run artifacts. The restricted path placeholder documents the authorized environment and is not an executable public-data contract.


### Configure the paired-statistics protocol

Imports metric and binomial-test utilities, defines the restricted project-root placeholder, identifies private silver and manual-audit inputs, and fixes the five contrasts, alpha level, bootstrap count, seeds, and metric order.


In [ ]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score, f1_score,
    matthews_corrcoef, roc_auc_score,
)


PROJECT_ROOT = Path('RESTRICTED_PROJECT_ROOT')
ROUND1_DIR = PROJECT_ROOT / 'journal_round1'
R1_08_ROOT = ROUND1_DIR / 'artifacts' / 'r1_08_repeated_seed_evaluation_v1'
HUMAN_DIR = ROUND1_DIR / 'reports' / 'human_label_audit' / 'r1_08_repeated_seed'
FINAL_AUDIT_PATH = ROUND1_DIR / 'data' / 'manual_audit_locked_test' / 'locked_test_audit_final_300_private.csv'
OUTPUT_DIR = ROUND1_DIR / 'reports' / 'final_model_statistics' / 'r1_08_repeated_seed'
BOOT_DIR = OUTPUT_DIR / 'bootstrap_replicates'
INDEX_DIR = OUTPUT_DIR / 'bootstrap_indices'

SILVER_PRED_PATH = R1_08_ROOT / 'locked_test_predictions_per_model_long_with_silver.csv'
SILVER_METRIC_PATH = R1_08_ROOT / 'locked_test_silver_metrics_per_seed.csv'
R1_08_MANIFEST_PATH = R1_08_ROOT / 'run_manifest.json'
R1_08_STATE_PATH = R1_08_ROOT / 'run_state.json'
R1_08_CONTRACT_PATH = R1_08_ROOT / 'prediction_contract.json'
HUMAN_PRED_PATH = HUMAN_DIR / 'human_label_evaluation_predictions_per_seed_private.csv'
HUMAN_METRIC_PATH = HUMAN_DIR / 'human_label_final_model_metrics_per_seed.csv'
HUMAN_SUMMARY_PATH = HUMAN_DIR / 'human_label_evaluation_summary.json'

EXPECTED_SEEDS = [42, 52, 62]
LABEL_ORDER = ['negatif', 'netral', 'positif']
LABEL_TO_ID = {label: index for index, label in enumerate(LABEL_ORDER)}
N_CLASSES = len(LABEL_ORDER)
MODEL_ORDER = [
    ('svm', 'SVM'), ('indobert', 'IndoBERT'), ('wa_svm_bert', 'WA (SVM+BERT)'),
    ('xgboost', 'XGBoost'), ('calibrated_xgboost', 'Calibrated XGBoost'),
]
MODEL_NAME = dict(MODEL_ORDER)
CONTRASTS = [
    ('K1', 'wa_svm_bert', 'svm', 'WA (SVM+BERT) versus SVM'),
    ('K2', 'wa_svm_bert', 'indobert', 'WA (SVM+BERT) versus IndoBERT'),
    ('K3', 'indobert', 'svm', 'IndoBERT versus SVM'),
    ('K4', 'calibrated_xgboost', 'wa_svm_bert', 'Calibrated XGBoost versus WA (SVM+BERT)'),
    ('K5', 'calibrated_xgboost', 'xgboost', 'Calibrated XGBoost versus XGBoost'),
]
ALPHA = 0.05
N_BOOTSTRAP = 2000
BOOTSTRAP_RANDOM_SEED = 20261030
METRIC_ATOL = 2e-7
PROTOCOL_VERSION = 'r1_02_protocol_v1_paired_statistics_holm_bootstrap'
METRIC_ORDER = ['accuracy', 'macro_f1', 'weighted_f1', 'balanced_accuracy', 'mcc', 'cohen_kappa', 'macro_roc_auc_ovr', 'ece', 'brier_score']

print('Output directory:', OUTPUT_DIR)
print('Seeds:', EXPECTED_SEEDS, '| contrasts:', [item[0] for item in CONTRASTS], '| bootstrap:', N_BOOTSTRAP)


### Define metric, paired-test, and bootstrap utilities

Defines checksum and persistence helpers, classification and calibration metrics, McNemar contingency calculations, Holm adjustment, and deterministic bootstrap procedures.


In [ ]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_json(path: Path, payload: dict) -> None:
    temporary = path.with_name(f'.{path.name}.tmp')
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=True, default=float), encoding='utf-8')
    temporary.replace(path)


def atomic_csv(path: Path, frame: pd.DataFrame) -> None:
    temporary = path.with_name(f'.{path.name}.tmp')
    frame.to_csv(temporary, index=False, encoding='utf-8')
    temporary.replace(path)


def normalize_label(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    return value if value in LABEL_TO_ID else np.nan


def prob_columns() -> list[str]:
    return ['prob_negatif', 'prob_netral', 'prob_positif']


def validate_probabilities(probabilities: np.ndarray, name: str) -> None:
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if probabilities.ndim != 2 or probabilities.shape[1] != N_CLASSES or not np.isfinite(probabilities).all():
        raise ValueError(f'{name}: invalid probability matrix.')
    if probabilities.min() < -1e-10 or probabilities.max() > 1 + 1e-10:
        raise ValueError(f'{name}: probabilities outside [0, 1].')
    if np.abs(probabilities.sum(axis=1) - 1.0).max() > 1e-5:
        raise ValueError(f'{name}: probabilities do not sum to one.')


def compute_ece(probabilities: np.ndarray, y_true: np.ndarray, sample_weight=None, n_bins: int = 10) -> float:
    weights = np.ones(len(y_true), dtype=float) if sample_weight is None else np.asarray(sample_weight, dtype=float)
    if len(weights) != len(y_true) or not np.isfinite(weights).all() or np.any(weights < 0) or weights.sum() <= 0:
        raise ValueError('Invalid ECE weights.')
    edges, total_weight, class_scores = np.linspace(0.0, 1.0, n_bins + 1), weights.sum(), []
    for class_id in range(N_CLASSES):
        probability, actual, score = probabilities[:, class_id], (y_true == class_id).astype(float), 0.0
        for left, right in zip(edges[:-1], edges[1:]):
            mask = (probability >= left) & ((probability <= right) if right == 1.0 else (probability < right))
            if mask.any():
                bin_weights = weights[mask]
                score += (bin_weights.sum() / total_weight) * abs(np.average(actual[mask], weights=bin_weights) - np.average(probability[mask], weights=bin_weights))
        class_scores.append(score)
    return float(np.mean(class_scores))


def compute_brier(probabilities: np.ndarray, y_true: np.ndarray, sample_weight=None) -> float:
    weights = np.ones(len(y_true), dtype=float) if sample_weight is None else np.asarray(sample_weight, dtype=float)
    return float(np.mean([np.average(((y_true == class_id).astype(float) - probabilities[:, class_id]) ** 2, weights=weights) for class_id in range(N_CLASSES)]))


def compute_metrics(y_true: np.ndarray, probabilities: np.ndarray, sample_weight=None) -> dict:
    y_true, probabilities = np.asarray(y_true, dtype=np.int64), np.asarray(probabilities, dtype=np.float64)
    validate_probabilities(probabilities, 'metric input')
    prediction = probabilities.argmax(axis=1)
    weights = None if sample_weight is None else np.asarray(sample_weight, dtype=float)
    return {
        'accuracy': float(accuracy_score(y_true, prediction, sample_weight=weights)),
        'macro_f1': float(f1_score(y_true, prediction, labels=np.arange(N_CLASSES), average='macro', sample_weight=weights, zero_division=0)),
        'weighted_f1': float(f1_score(y_true, prediction, labels=np.arange(N_CLASSES), average='weighted', sample_weight=weights, zero_division=0)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, prediction, sample_weight=weights)),
        'mcc': float(matthews_corrcoef(y_true, prediction, sample_weight=weights)),
        'cohen_kappa': float(cohen_kappa_score(y_true, prediction, labels=np.arange(N_CLASSES), sample_weight=weights)),
        'macro_roc_auc_ovr': float(roc_auc_score(y_true, probabilities, labels=np.arange(N_CLASSES), multi_class='ovr', average='macro', sample_weight=weights)),
        'ece': compute_ece(probabilities, y_true, weights),
        'brier_score': compute_brier(probabilities, y_true, weights),
    }


def holm_adjust(raw_pvalues: np.ndarray, alpha: float = ALPHA):
    raw_pvalues = np.asarray(raw_pvalues, dtype=float)
    if raw_pvalues.ndim != 1 or len(raw_pvalues) == 0 or not np.isfinite(raw_pvalues).all():
        raise ValueError('Holm requires finite p-values.')
    m, order = len(raw_pvalues), np.argsort(raw_pvalues, kind='mergesort')
    sorted_p = raw_pvalues[order]
    threshold_sorted = alpha / (m - np.arange(m))
    adjusted_sorted = np.maximum.accumulate(np.minimum(1.0, sorted_p * (m - np.arange(m))))
    reject_sorted = np.zeros(m, dtype=bool)
    for index, value in enumerate(sorted_p):
        if value <= threshold_sorted[index] and (index == 0 or reject_sorted[index - 1]):
            reject_sorted[index] = True
        else:
            break
    adjusted, rank, threshold, reject = np.empty(m), np.empty(m, dtype=int), np.empty(m), np.empty(m, dtype=bool)
    adjusted[order], rank[order], threshold[order], reject[order] = adjusted_sorted, np.arange(1, m + 1), threshold_sorted, reject_sorted
    return adjusted, rank, threshold, reject


def stratified_indices(y_true: np.ndarray, n_bootstrap: int, random_seed: int) -> np.ndarray:
    rng, pieces = np.random.default_rng(random_seed), []
    for class_id in range(N_CLASSES):
        rows = np.flatnonzero(y_true == class_id)
        if len(rows) == 0:
            raise ValueError(f'Bootstrap target lacks class {class_id}.')
        pieces.append(rng.choice(rows, size=(n_bootstrap, len(rows)), replace=True))
    return np.concatenate(pieces, axis=1).astype(np.int32, copy=False)


def read_bootstrap_cache(path: Path):
    if not path.is_file():
        return None
    with np.load(path, allow_pickle=False) as artifact:
        names, values = artifact['metric_names'].astype(str).tolist(), artifact['values']
    if names != METRIC_ORDER or values.shape != (N_BOOTSTRAP, len(METRIC_ORDER)):
        raise RuntimeError(f'Incompatible bootstrap cache: {path}')
    return values.astype(np.float64, copy=False)


def write_bootstrap_cache(path: Path, values: np.ndarray) -> None:
    temporary = path.with_name(f'.{path.name}.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, metric_names=np.asarray(METRIC_ORDER), values=values)
    temporary.replace(path)


def bootstrap_metrics(y_true, probabilities, indices, sample_weight=None) -> np.ndarray:
    values, weights = np.empty((len(indices), len(METRIC_ORDER))), None if sample_weight is None else np.asarray(sample_weight, dtype=float)
    for replicate, rows in enumerate(indices):
        sampled_weights = None if weights is None else weights[rows]
        metrics = compute_metrics(y_true[rows], probabilities[rows], sampled_weights)
        values[replicate] = [metrics[name] for name in METRIC_ORDER]
    return values


### Load and validate silver and manual evaluation inputs

Checks the completed repeated-seed prediction contract, the private manual-label evaluation linkage, row alignment, seed and model order, checksums, and required inputs before statistical calculations.


In [ ]:
required_paths = [SILVER_PRED_PATH, SILVER_METRIC_PATH, R1_08_MANIFEST_PATH, R1_08_STATE_PATH, R1_08_CONTRACT_PATH, HUMAN_PRED_PATH, HUMAN_METRIC_PATH, HUMAN_SUMMARY_PATH, FINAL_AUDIT_PATH]
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing paired-statistics input(s): ' + '; '.join(missing))

r1_08_manifest = json.loads(R1_08_MANIFEST_PATH.read_text(encoding='utf-8'))
r1_08_state = json.loads(R1_08_STATE_PATH.read_text(encoding='utf-8'))
r1_08_contract = json.loads(R1_08_CONTRACT_PATH.read_text(encoding='utf-8'))
human_summary = json.loads(HUMAN_SUMMARY_PATH.read_text(encoding='utf-8'))
if r1_08_state.get('status') != 'completed' or r1_08_manifest.get('locked_test_metrics_computed') is not True:
    raise RuntimeError('Repeated-seed evaluation is not complete.')
if r1_08_contract.get('seeds') != EXPECTED_SEEDS:
    raise RuntimeError('Repeated-seed seed contract does not match paired-statistics input.')
contract_models = [(item.get('model_key'), item.get('model')) for item in r1_08_contract.get('model_order', [])]
if contract_models != MODEL_ORDER:
    raise RuntimeError('Repeated-seed model contract does not match paired-statistics input.')
if r1_08_manifest.get('output_checksums', {}).get(SILVER_PRED_PATH.name) != sha256_file(SILVER_PRED_PATH):
    raise RuntimeError('Repeated-seed prediction checksum does not match its manifest.')
if human_summary.get('prediction_source_sha256') != sha256_file(SILVER_PRED_PATH):
    raise RuntimeError('Human-label evaluation is not linked to the current repeated-seed predictions.')

silver_long = pd.read_csv(SILVER_PRED_PATH)
human_long = pd.read_csv(HUMAN_PRED_PATH)
final_audit = pd.read_csv(FINAL_AUDIT_PATH)
silver_historical = pd.read_csv(SILVER_METRIC_PATH)
human_historical = pd.read_csv(HUMAN_METRIC_PATH)

base_columns = {'seed', 'row_id', 'model_key', 'model', 'prediction', 'prediction_label', 'confidence', 'entropy', *prob_columns()}
required_silver = base_columns | {'y_silver', 'y_silver_label'}
required_human = base_columns | {'audit_case_id', 'audit_origin', 'silver_label_lstm', 'final_manual_label', 'design_weight'}
for name, frame, required, expected_n, target_column in [
    ('silver', silver_long, required_silver, 1172, 'y_silver_label'),
    ('human', human_long, required_human, 300, 'final_manual_label'),
]:
    absent = sorted(required.difference(frame.columns))
    if absent:
        raise ValueError(f'{name} prediction columns missing: {absent}')
    if sorted(frame['seed'].unique().tolist()) != EXPECTED_SEEDS or set(frame['model_key']) != set(MODEL_NAME):
        raise RuntimeError(f'{name} prediction has unexpected seeds or models.')
    if len(frame) != expected_n * len(EXPECTED_SEEDS) * len(MODEL_ORDER) or frame.duplicated(['seed', 'row_id', 'model_key']).any():
        raise RuntimeError(f'{name} prediction key/count is invalid.')
    frame['prediction_label'] = frame['prediction_label'].map(normalize_label)
    frame[target_column] = frame[target_column].map(normalize_label)
    if frame[['prediction_label', target_column]].isna().any().any():
        raise ValueError(f'{name} contains an invalid prediction or reference label.')
    probabilities = frame[prob_columns()].to_numpy(dtype=float)
    validate_probabilities(probabilities, name)
    if not np.array_equal(frame['prediction'].to_numpy(dtype=np.int64), probabilities.argmax(axis=1)):
        raise RuntimeError(f'{name} hard predictions do not equal probability argmax.')
    for seed in EXPECTED_SEEDS:
        ids, labels = None, None
        for model_key, _ in MODEL_ORDER:
            group = frame.loc[(frame['seed'] == seed) & (frame['model_key'] == model_key)].sort_values('row_id')
            row_ids, target = group['row_id'].to_numpy(dtype=np.int64), group[target_column].to_numpy(dtype=str)
            if len(group) != expected_n:
                raise RuntimeError(f'{name}/{seed}/{model_key} is incomplete.')
            if ids is None:
                ids, labels = row_ids, target
            elif not np.array_equal(ids, row_ids) or not np.array_equal(labels, target):
                raise RuntimeError(f'{name}/{seed} is not aligned across models.')

silver_long['y_silver_label'] = silver_long['y_silver_label'].map(normalize_label)
if not np.array_equal(silver_long['y_silver'].to_numpy(dtype=np.int64), silver_long['y_silver_label'].map(LABEL_TO_ID).to_numpy(dtype=np.int64)):
    raise RuntimeError('Repeated-seed numeric and text silver labels disagree.')
human_long['silver_label_lstm'] = human_long['silver_label_lstm'].map(normalize_label)
audit_reference = final_audit[['locked_test_row', 'silver_label_lstm', 'final_manual_label', 'design_weight']].rename(columns={'locked_test_row': 'row_id'}).copy()
for column in ['silver_label_lstm', 'final_manual_label']:
    audit_reference[column] = audit_reference[column].map(normalize_label)
if len(audit_reference) != 300 or audit_reference['row_id'].duplicated().any() or audit_reference.isna().any().any():
    raise RuntimeError('Final manual audit is incomplete or invalid.')
human_reference = human_long[['row_id', 'silver_label_lstm', 'final_manual_label', 'design_weight']].drop_duplicates('row_id')
audit_check = human_reference.merge(audit_reference, on='row_id', suffixes=('_prediction', '_audit'), validate='one_to_one')
if len(audit_check) != 300 or not audit_check['silver_label_lstm_prediction'].eq(audit_check['silver_label_lstm_audit']).all() or not audit_check['final_manual_label_prediction'].eq(audit_check['final_manual_label_audit']).all() or not np.allclose(audit_check['design_weight_prediction'], audit_check['design_weight_audit']):
    raise RuntimeError('Human-label predictions do not match the final manual-audit reference.')

INPUT_HASHES = {str(path.relative_to(PROJECT_ROOT)): sha256_file(path) for path in [SILVER_PRED_PATH, R1_08_MANIFEST_PATH, R1_08_CONTRACT_PATH, HUMAN_PRED_PATH, HUMAN_SUMMARY_PATH, FINAL_AUDIT_PATH]}
print('Validated', len(silver_long), 'silver rows and', len(human_long), 'human-audit rows.')


### Recompute metrics and verify source consistency

Recalculates model metrics for locked-test silver labels and manual audit labels, including design-weighted manual metrics, and verifies agreement with the upstream aggregate evaluation results.


In [ ]:
def metric_rows(frame, reference_target, target_column, weight_specs):
    rows = []
    for seed in EXPECTED_SEEDS:
        for model_key, model_name in MODEL_ORDER:
            group = frame.loc[(frame['seed'] == seed) & (frame['model_key'] == model_key)].sort_values('row_id')
            y_true, probabilities = group[target_column].map(LABEL_TO_ID).to_numpy(dtype=np.int64), group[prob_columns()].to_numpy(dtype=float)
            for weighting, weight_column in weight_specs:
                weights = None if weight_column is None else group[weight_column].to_numpy(dtype=float)
                for metric, value in compute_metrics(y_true, probabilities, weights).items():
                    rows.append({'reference_target': reference_target, 'weighting': weighting, 'seed': seed, 'model_key': model_key, 'model': model_name, 'n_reviews': len(group), 'metric': metric, 'value': value})
    return rows

metric_records = metric_rows(silver_long, 'locked_test_silver', 'y_silver_label', [('unweighted', None)])
metric_records += metric_rows(human_long, 'human_audit_manual', 'final_manual_label', [('unweighted', None), ('design_weighted', 'design_weight')])
metrics_per_seed = pd.DataFrame(metric_records).sort_values(['reference_target', 'weighting', 'model_key', 'seed', 'metric']).reset_index(drop=True)

checks = []
for seed in EXPECTED_SEEDS:
    for model_key, _ in MODEL_ORDER:
        silver_new = metrics_per_seed.loc[(metrics_per_seed['reference_target'] == 'locked_test_silver') & (metrics_per_seed['weighting'] == 'unweighted') & (metrics_per_seed['seed'] == seed) & (metrics_per_seed['model_key'] == model_key)].set_index('metric')['value']
        silver_old = silver_historical.loc[(silver_historical['seed'] == seed) & (silver_historical['model_key'] == model_key)]
        if len(silver_old) != 1:
            raise RuntimeError(f'Missing repeated-seed metric row for {seed}/{model_key}.')
        for metric in ['accuracy', 'macro_f1', 'ece', 'brier_score']:
            difference = abs(float(silver_new[metric]) - float(silver_old.iloc[0][metric]))
            checks.append({'source': 'repeated_seed_evaluation', 'seed': seed, 'model_key': model_key, 'metric': metric, 'absolute_difference': difference})
            if difference > METRIC_ATOL:
                raise RuntimeError(f'Repeated-seed metric mismatch for {seed}/{model_key}/{metric}: {difference:.3e}')
        human_old = human_historical.loc[(human_historical['seed'] == seed) & (human_historical['model_key'] == model_key)]
        if len(human_old) != 1:
            raise RuntimeError(f'Missing human-label metric row for {seed}/{model_key}.')
        for weighting, suffix, metric_names in [('unweighted', 'unweighted', ['accuracy', 'macro_f1', 'balanced_accuracy']), ('design_weighted', 'design_weighted', ['accuracy', 'macro_f1'])]:
            human_new = metrics_per_seed.loc[(metrics_per_seed['reference_target'] == 'human_audit_manual') & (metrics_per_seed['weighting'] == weighting) & (metrics_per_seed['seed'] == seed) & (metrics_per_seed['model_key'] == model_key)].set_index('metric')['value']
            for metric in metric_names:
                difference = abs(float(human_new[metric]) - float(human_old.iloc[0][f'{metric}_{suffix}']))
                checks.append({'source': 'human_label_evaluation', 'seed': seed, 'model_key': model_key, 'metric': f'{metric}_{suffix}', 'absolute_difference': difference})
                if difference > METRIC_ATOL:
                    raise RuntimeError(f'Human-label metric mismatch for {seed}/{model_key}/{metric}_{suffix}: {difference:.3e}')

reproduction_checks = pd.DataFrame(checks)
metric_summary = metrics_per_seed.groupby(['reference_target', 'weighting', 'model_key', 'model', 'metric'])['value'].agg(['mean', 'std', 'min', 'max']).reset_index().rename(columns={'mean': 'mean_across_seeds', 'std': 'sample_sd_across_seeds'})
atomic_csv(OUTPUT_DIR / 'final_model_metrics_recomputed_per_seed.csv', metrics_per_seed)
atomic_csv(OUTPUT_DIR / 'final_model_metric_repeated_seed_summary.csv', metric_summary)
atomic_csv(OUTPUT_DIR / 'metric_reproduction_checks.csv', reproduction_checks)
print('Metric reproduction checks passed:', len(reproduction_checks))


### Run paired McNemar comparisons with Holm adjustment

Evaluates the five pre-specified model contrasts on aligned predictions for each seed and target, records contingency counts and raw p-values, then applies Holm correction within the stated comparison family.


In [ ]:
def aligned_predictions(frame, seed, target_column):
    y_true, prediction_by_model = None, {}
    for model_key, _ in MODEL_ORDER:
        group = frame.loc[(frame['seed'] == seed) & (frame['model_key'] == model_key)].sort_values('row_id')
        target, prediction = group[target_column].map(LABEL_TO_ID).to_numpy(dtype=np.int64), group['prediction'].to_numpy(dtype=np.int64)
        if y_true is None:
            y_true = target
        elif not np.array_equal(y_true, target):
            raise RuntimeError(f'Target labels are not aligned for seed {seed}.')
        prediction_by_model[model_key] = prediction
    return y_true, prediction_by_model

mcnemar_rows = []
for reference_target, frame, target_column in [('locked_test_silver', silver_long, 'y_silver_label'), ('human_audit_manual', human_long, 'final_manual_label')]:
    for seed in EXPECTED_SEEDS:
        y_true, predictions = aligned_predictions(frame, seed, target_column)
        for contrast_id, model_a_key, model_b_key, contrast in CONTRASTS:
            a_correct, b_correct = predictions[model_a_key] == y_true, predictions[model_b_key] == y_true
            both_correct = int(np.sum(a_correct & b_correct))
            a_only = int(np.sum(a_correct & ~b_correct))
            b_only = int(np.sum(~a_correct & b_correct))
            both_wrong = int(np.sum(~a_correct & ~b_correct))
            discordant = a_only + b_only
            raw_pvalue = 1.0 if discordant == 0 else float(binomtest(a_only, n=discordant, p=0.5, alternative='two-sided').pvalue)
            mcnemar_rows.append({'reference_target': reference_target, 'seed': seed, 'contrast_id': contrast_id, 'contrast': contrast, 'model_a_key': model_a_key, 'model_a': MODEL_NAME[model_a_key], 'model_b_key': model_b_key, 'model_b': MODEL_NAME[model_b_key], 'n_reviews': len(y_true), 'both_correct': both_correct, 'a_only_correct': a_only, 'b_only_correct': b_only, 'both_incorrect': both_wrong, 'discordant_correctness': discordant, 'delta_accuracy_a_minus_b': (a_only - b_only) / len(y_true), 'mcnemar_method': 'exact_two_sided_binomial', 'raw_pvalue': raw_pvalue})

mcnemar_results = pd.DataFrame(mcnemar_rows)
for reference_target in mcnemar_results['reference_target'].unique():
    mask = mcnemar_results['reference_target'].eq(reference_target)
    adjusted, rank, threshold, rejected = holm_adjust(mcnemar_results.loc[mask, 'raw_pvalue'].to_numpy(), ALPHA)
    mcnemar_results.loc[mask, 'holm_family_size'] = int(mask.sum())
    mcnemar_results.loc[mask, 'holm_rank'] = rank
    mcnemar_results.loc[mask, 'holm_step_threshold'] = threshold
    mcnemar_results.loc[mask, 'holm_adjusted_pvalue'] = adjusted
    mcnemar_results.loc[mask, 'holm_reject_at_alpha_0_05'] = rejected

direction_rows = []
for (reference_target, contrast_id, contrast), group in mcnemar_results.groupby(['reference_target', 'contrast_id', 'contrast'], sort=False):
    delta = group['delta_accuracy_a_minus_b'].to_numpy(dtype=float)
    direction_rows.append({'reference_target': reference_target, 'contrast_id': contrast_id, 'contrast': contrast, 'seeds': ','.join(str(seed) for seed in sorted(group['seed'].astype(int))), 'mean_delta_accuracy_a_minus_b': delta.mean(), 'sample_sd_delta_accuracy_a_minus_b': delta.std(ddof=1), 'seeds_a_higher': int((delta > 0).sum()), 'seeds_b_higher': int((delta < 0).sum()), 'seeds_tied': int((delta == 0).sum()), 'minimum_holm_adjusted_pvalue': group['holm_adjusted_pvalue'].min(), 'any_holm_rejection': bool(group['holm_reject_at_alpha_0_05'].any())})

paired_direction_summary = pd.DataFrame(direction_rows)
atomic_csv(OUTPUT_DIR / 'mcnemar_contingency_per_seed.csv', mcnemar_results)
atomic_csv(OUTPUT_DIR / 'mcnemar_results_holm_per_target.csv', mcnemar_results)
atomic_csv(OUTPUT_DIR / 'paired_comparison_seed_direction_summary.csv', paired_direction_summary)
print('McNemar tests:', len(mcnemar_results), '= 5 contrasts x 3 seeds x 2 targets.')
display(mcnemar_results[['reference_target', 'seed', 'contrast_id', 'a_only_correct', 'b_only_correct', 'delta_accuracy_a_minus_b', 'raw_pvalue', 'holm_adjusted_pvalue', 'holm_reject_at_alpha_0_05']])


### Generate deterministic bootstrap intervals

Produces repeated bootstrap estimates for the relevant model and metric combinations, including design-weighted manual-audit metrics, while retaining replicate files as private artifacts.


In [ ]:
bootstrap_specs = [
    ('locked_test_silver', silver_long, 'y_silver_label', 'unweighted', None, 0),
    ('human_audit_manual', human_long, 'final_manual_label', 'unweighted', None, 1),
    ('human_audit_manual', human_long, 'final_manual_label', 'design_weighted', 'design_weight', 1),
]
state_path = OUTPUT_DIR / 'bootstrap_run_state.json'
bootstrap_state = {'protocol_version': PROTOCOL_VERSION, 'n_bootstrap': N_BOOTSTRAP, 'metric_order': METRIC_ORDER, 'completed_groups': [], 'updated_at_utc': utc_now()}
if state_path.is_file():
    bootstrap_state = json.loads(state_path.read_text(encoding='utf-8'))
    if bootstrap_state.get('protocol_version') != PROTOCOL_VERSION or bootstrap_state.get('n_bootstrap') != N_BOOTSTRAP or bootstrap_state.get('metric_order') != METRIC_ORDER:
        raise RuntimeError('Existing bootstrap state belongs to a different protocol.')

bootstrap_ci_rows = []
for reference_target, frame, target_column, weighting, weight_column, random_offset in bootstrap_specs:
    base_group = frame.loc[(frame['seed'] == EXPECTED_SEEDS[0]) & (frame['model_key'] == MODEL_ORDER[0][0])].sort_values('row_id')
    base_y = base_group[target_column].map(LABEL_TO_ID).to_numpy(dtype=np.int64)
    index_path = INDEX_DIR / f'{reference_target}_indices.npz'
    if index_path.is_file():
        with np.load(index_path, allow_pickle=False) as artifact:
            indices = artifact['indices']
        if indices.shape != (N_BOOTSTRAP, len(base_y)):
            raise RuntimeError(f'Incompatible bootstrap index cache: {index_path}')
    else:
        indices = stratified_indices(base_y, N_BOOTSTRAP, BOOTSTRAP_RANDOM_SEED + random_offset)
        temporary = index_path.with_name(f'.{index_path.name}.tmp')
        with temporary.open('wb') as handle:
            np.savez_compressed(handle, indices=indices)
        temporary.replace(index_path)

    for seed in EXPECTED_SEEDS:
        for model_key, model_name in MODEL_ORDER:
            group = frame.loc[(frame['seed'] == seed) & (frame['model_key'] == model_key)].sort_values('row_id')
            y_true, probabilities = group[target_column].map(LABEL_TO_ID).to_numpy(dtype=np.int64), group[prob_columns()].to_numpy(dtype=float)
            if not np.array_equal(y_true, base_y):
                raise RuntimeError(f'Bootstrap target changed for {reference_target}/{seed}/{model_key}.')
            weights = None if weight_column is None else group[weight_column].to_numpy(dtype=float)
            cache_path = BOOT_DIR / f'{reference_target}__{weighting}__seed_{seed}__{model_key}.npz'
            cache = read_bootstrap_cache(cache_path)
            group_name = f'{reference_target}/{weighting}/seed_{seed}/{model_key}'
            if cache is None:
                print('Computing:', group_name)
                cache = bootstrap_metrics(y_true, probabilities, indices, weights)
                write_bootstrap_cache(cache_path, cache)
            else:
                print('Reusing:', group_name)
            point = compute_metrics(y_true, probabilities, weights)
            quantiles = np.quantile(cache, [0.025, 0.5, 0.975], axis=0)
            for metric_index, metric in enumerate(METRIC_ORDER):
                bootstrap_ci_rows.append({'reference_target': reference_target, 'weighting': weighting, 'seed': seed, 'model_key': model_key, 'model': model_name, 'n_reviews': len(group), 'metric': metric, 'point_estimate': point[metric], 'bootstrap_p2_5': quantiles[0, metric_index], 'bootstrap_p50': quantiles[1, metric_index], 'bootstrap_p97_5': quantiles[2, metric_index], 'n_bootstrap': N_BOOTSTRAP})
            if group_name not in bootstrap_state['completed_groups']:
                bootstrap_state['completed_groups'].append(group_name)
                bootstrap_state['updated_at_utc'] = utc_now()
                atomic_json(state_path, bootstrap_state)

bootstrap_ci = pd.DataFrame(bootstrap_ci_rows).sort_values(['reference_target', 'weighting', 'model_key', 'seed', 'metric']).reset_index(drop=True)
atomic_csv(OUTPUT_DIR / 'bootstrap_confidence_intervals_per_seed.csv', bootstrap_ci)
print('Bootstrap CI rows written:', len(bootstrap_ci))


### Write statistical summaries and completion record

Writes aggregate metric, McNemar-Holm, paired-comparison, and confidence-interval tables plus a summary manifest. The stage does not fit a model, select a seed, or average probabilities.


In [ ]:
output_paths = [
    OUTPUT_DIR / 'final_model_metrics_recomputed_per_seed.csv',
    OUTPUT_DIR / 'final_model_metric_repeated_seed_summary.csv',
    OUTPUT_DIR / 'metric_reproduction_checks.csv',
    OUTPUT_DIR / 'mcnemar_contingency_per_seed.csv',
    OUTPUT_DIR / 'mcnemar_results_holm_per_target.csv',
    OUTPUT_DIR / 'paired_comparison_seed_direction_summary.csv',
    OUTPUT_DIR / 'bootstrap_confidence_intervals_per_seed.csv',
]
missing_outputs = [path.name for path in output_paths if not path.is_file()]
if missing_outputs:
    raise RuntimeError('Cannot finalize paired statistics; missing: ' + ', '.join(missing_outputs))

manifest = {
    'protocol_version': PROTOCOL_VERSION, 'completed_at_utc': utc_now(), 'seeds': EXPECTED_SEEDS,
    'models': [{'model_key': key, 'model': name} for key, name in MODEL_ORDER],
    'contrasts': [{'contrast_id': cid, 'model_a_key': a, 'model_b_key': b, 'contrast': name} for cid, a, b, name in CONTRASTS],
    'mcnemar': {'method': 'exact_two_sided_binomial', 'alpha': ALPHA, 'multiple_comparison_method': 'Holm-Bonferroni', 'family_size_per_reference_target': len(CONTRASTS) * len(EXPECTED_SEEDS), 'no_cross_seed_pooling': True},
    'bootstrap': {'n_replicates': N_BOOTSTRAP, 'random_seed': BOOTSTRAP_RANDOM_SEED, 'method': 'stratified percentile bootstrap per model and seed', 'metric_order': METRIC_ORDER},
    'input_hashes': INPUT_HASHES,
    'guardrails': ['Silver locked-test metrics are agreement with LSTM-derived silver labels.', 'Human audit covers 300 stratified locked-test reviews, not a complete human-labelled test set.', 'No seed selection or probability averaging occurred.'],
    'output_checksums': {path.name: sha256_file(path) for path in output_paths},
}
atomic_json(OUTPUT_DIR / 'analysis_plan_and_manifest.json', manifest)
atomic_json(OUTPUT_DIR / 'input_integrity_summary.json', {'silver_prediction_rows': len(silver_long), 'human_prediction_rows': len(human_long), 'manual_audit_reviews': len(audit_reference), 'metric_reproduction_checks': len(reproduction_checks), 'maximum_metric_difference': float(reproduction_checks['absolute_difference'].max()), 'input_hashes': INPUT_HASHES})
atomic_json(OUTPUT_DIR / 'r1_02_summary.json', {'status': 'completed', 'completed_at_utc': utc_now(), 'mcnemar_tests': len(mcnemar_results), 'holm_rejections_by_target': {target: int(group['holm_reject_at_alpha_0_05'].sum()) for target, group in mcnemar_results.groupby('reference_target')}, 'bootstrap_ci_rows': len(bootstrap_ci), 'note': 'No model fitting, seed selection, or probability averaging occurred.'})
atomic_json(OUTPUT_DIR / 'run_state.json', {'protocol_version': PROTOCOL_VERSION, 'status': 'completed', 'completed_at_utc': utc_now(), 'completed_stages': ['input_validation', 'metric_reproduction', 'mcnemar_holm', 'bootstrap_ci', 'finalization']})
print('Paired statistics completed:', OUTPUT_DIR)
print('Report per-seed values, mean +/- sample SD, and Holm-adjusted p-values. Do not select a best seed.')
